# 9. Mały Transformer od podstaw: PyTorch

**Cel:** zobaczyć embedding, kod pozycji, self-attention i klasyfikację krótkich sekwencji. Dane lokalne, bez pobierania wag. Czas: 20–25 min. Wymaga `torch`.

**Uruchamianie:** wykonuj komórki kolejno. Dane są generowane lokalnie; nie jest potrzebny internet.

### Zadanie
Sekwencja ma długość 5, słownik 1–9. Klasa 1 oznacza wystąpienie tokenu 9. Architektura: embedding + embedding pozycji + `TransformerEncoderLayer` + pooling + klasyfikator. `d_model=16`, `nhead=2`.

In [ ]:
import torch
from torch import nn
torch.manual_seed(7)
torch.set_num_threads(1)
def dataset(n,seed):
    gen=torch.Generator().manual_seed(seed)
    tokens=torch.randint(1,9,(n,5),generator=gen)
    labels=torch.arange(n)%2
    positions=torch.randint(0,5,(n//2,),generator=gen)
    indices=torch.arange(1,n,2)
    tokens[indices,positions]=9
    order=torch.randperm(n,generator=gen)
    return tokens[order],labels[order]
train_x,train_y=dataset(160,8);test_x,test_y=dataset(80,9)
assert torch.equal((train_x==9).any(1).long(),train_y)


In [ ]:
class TinyTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.token=nn.Embedding(10,16)
        self.position=nn.Embedding(5,16)
        layer=nn.TransformerEncoderLayer(d_model=16,nhead=2,dim_feedforward=32,
                                         dropout=0,batch_first=True)
        self.encoder=nn.TransformerEncoder(layer,num_layers=1,enable_nested_tensor=False)
        self.head=nn.Linear(16,2)
    def forward(self,x):
        pos=torch.arange(x.shape[1],device=x.device)
        embedded=self.token(x)+self.position(pos)[None,:,:]
        return self.head(self.encoder(embedded).mean(dim=1))
model=TinyTransformer()
opt=torch.optim.AdamW(model.parameters(),lr=0.008)
for step in range(90):
    model.train();opt.zero_grad()
    logits=model(train_x)
    loss=nn.functional.cross_entropy(logits,train_y)
    loss.backward();opt.step()
    if step in (0,29,89):print('iteracja',step+1,'strata',round(float(loss.item()),4))
model.eval()
with torch.no_grad():
    accuracy=float((model(test_x).argmax(-1)==test_y).float().mean())
print('Dokładność testowa:',round(accuracy,3),'parametry:',sum(p.numel() for p in model.parameters()))
assert accuracy>0.90


**Analiza:** Zastąp kody pozycji zerami albo zmień `nhead` na 1. Czy przy tym zadaniu, które zależy od obecności słowa, pozycja pomaga? `TransformerEncoderLayer` oblicza uwagę; wynik to klasyfikacja całej sekwencji, nie generowanie tekstu.